<a href="https://colab.research.google.com/github/team0243/Project_ML/blob/main/KS_RevisedCode_RCC_UTUC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sklearn
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import imblearn
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import RFE
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, roc_auc_score
from sklearn.preprocessing import MinMaxScaler
import time
from sklearn.metrics import precision_score, recall_score,f1_score
import warnings


In [ ]:
!python --version   #check python version

In [ ]:
print("Scikit-learn version:", sklearn.__version__)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("imblearn version:", imblearn.__version__)
print("Seaborn version:", sns.__version__)

## Loading and Checking the dataset


---

This file contains CBC (Complete Blood Count) data of patients, which is used for pathological studies. It focuses on the classification of Renal Cell Carcinoma (RCC) and Upper Tract Urothelial Carcinoma (UTUC). This data may be used for developing Machine Learning models for medical analysis.


In [ ]:
url = 'https://github.com/team0243/Project_ML/blob/main/Dataset_CBC_RCC_UTUC.xlsx?raw=true'
try:
  df = pd.read_excel(url)
except Exception as e:
  print(f"An error occurred: {e}")
  print("Please ensure the link is correct, the file exists, and the proper permissions are set.")

In [ ]:
# Prints information about a DataFrame
df.info()

In [ ]:
df.describe(include="all")

In [ ]:
df.head()

**Checkingfor duplicates and MIssing Value**

In [ ]:
#count duplicates  in DataFrame
print(df.duplicated().sum())

In [ ]:
# Count NaN values in DataFrame
df.isna().sum()

In [ ]:
sns.pairplot(df[['Age ', 'NLR', 'PLR', 'WBC','PLT','NE%','LY%', 'Diagnosis']], hue='Diagnosis')
plt.figure(figsize=(10, 8),dpi = 600)
plt.show()

**Checking the balance between RCC and UTUC**

In [ ]:
df['Diagnosis'].value_counts().plot.bar()

In [ ]:
df['Diagnosis'].value_counts(normalize=True)

**The ratio between RCC and UTUC data is noticeably imbalanced. Therefore, we will address the imbalance issue using over-sampling.**


---

All Features used to prepare data for Machine Learning Models

In [ ]:
X = df.drop(columns=['Diagnosis'])  # Independent Variables
y = df['Diagnosis'] # Target or Label to be predicted

In [ ]:
X.head(5)

In [ ]:
X.shape, y.shape

In [ ]:
#Split data

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=25,
    stratify=y  # เพิ่ม stratify
)


In [ ]:
#SMOTE For Training set

sm = SMOTE(sampling_strategy=0.90, random_state=25, k_neighbors=5)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print(f"Training set after SMOTE: {X_train_res.shape}")
print(f"Test set (original): {X_test.shape}")


In [ ]:
# GridSearchCV , Best Hyperparameters
model_rf = RandomForestClassifier(random_state=25)

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

cv_strategy = StratifiedKFold(
    n_splits=5,      # 5-fold
    shuffle=True,
    random_state=25
)

grid_rf = GridSearchCV(
    model_rf,
    param_grid,
    cv=cv_strategy,        #  StratifiedKFold replace cv=5
    scoring='f1_macro',    # replace accuracy → f1_macro
    n_jobs=-1,
    verbose=2,
    refit=True             # refit , best params on training set
)

grid_rf.fit(X_train_res, y_train_res)  # fit  SMOTE data

print("Best Hyperparameters:", grid_rf.best_params_)
print("Best CV F1-score:", grid_rf.best_score_)

best_params = grid_rf.best_params_


In [ ]:
# RFE + Cross-Validation
n_features_range = [2, 4, 5, 6, 7]
cv_rfe = StratifiedKFold(n_splits=5, shuffle=True, random_state=25)

results = []

for n_features in n_features_range:

    # สร้าง RFE ด้วย best hyperparameters จาก GridSearchCV
    rfe = RFE(
        estimator=RandomForestClassifier(**best_params, random_state=25),
        n_features_to_select=n_features,
        step=1
    )

    # Cross-validate RFE pipeline
    cv_scores = cross_validate(
        rfe,
        X_train_res,
        y_train_res,
        cv=cv_rfe,
        scoring={
            'precision': 'precision_macro',
            'recall': 'recall_macro',
            'f1': 'f1_macro',
            'accuracy': 'accuracy'
        },
        return_train_score=False
    )

    # correct result
    results.append({
        'n_features': n_features,
        'accuracy_mean': cv_scores['test_accuracy'].mean(),
        'accuracy_std': cv_scores['test_accuracy'].std(),
        'precision_mean': cv_scores['test_precision'].mean(),
        'precision_std': cv_scores['test_precision'].std(),
        'recall_mean': cv_scores['test_recall'].mean(),
        'recall_std': cv_scores['test_recall'].std(),
        'f1_mean': cv_scores['test_f1'].mean(),
        'f1_std': cv_scores['test_f1'].std()
    })

    print(f"\nn_features={n_features}: "
          f"F1={cv_scores['test_f1'].mean():.4f} "
          f"(±{cv_scores['test_f1'].std():.4f})")

In [ ]:
# Select Optimal Number of Features
results_df = pd.DataFrame(results)
print("\n" + "="*60)
print("Feature Selection Results (5-Fold Stratified CV)")
print("="*60)
print(results_df[['n_features',
                   'accuracy_mean',
                   'precision_mean',
                   'recall_mean',
                   'f1_mean',
                   'f1_std']].to_string(index=False))

# select n_features for F1 maximum
best_n_features = results_df.loc[
    results_df['f1_mean'].idxmax(),
    'n_features'
]
print(f"\nOptimal number of features: {best_n_features}")

In [ ]:
#  Train Final Model ด้วย Optimal Features

final_rfe = RFE(
    estimator=RandomForestClassifier(**best_params, random_state=25),
    n_features_to_select=best_n_features,
    step=1
)

# Fit RFE  training set
final_rfe.fit(X_train_res, y_train_res)

# show features to select
selected_features = X.columns[final_rfe.support_].tolist()
print(f"\nSelected Features: {selected_features}")


In [ ]:
# Evaluate บน Test Set
# Transform test set by RFE fit
X_test_rfe = final_rfe.transform(X_test)
X_train_rfe = final_rfe.transform(X_train_res)

# Train final classifier
final_rf = RandomForestClassifier(**best_params, random_state=25)
final_rf.fit(X_train_rfe, y_train_res)

# Predict
y_pred = final_rf.predict(X_test_rfe)

# Results
print("\n" + "="*60)
print(f"Final Model Performance (n_features={best_n_features})")
print("="*60)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['RCC', 'UTUC']))

## Select Best Feature

In [ ]:
X = df[['Age ', 'NLR', 'PLR', 'WBC', 'PLT', 'NE%', 'LY%']]  # Independent Variables five paramerers
y = df['Diagnosis'] # Target or Label to be predicted

Data visualization using pair plot and box plot for five parameters.

In [ ]:
#Display piarplot all five parameters
pair = sns.pairplot(df[['Age ', 'NLR', 'PLR', 'WBC', 'PLT', 'NE%', 'LY%', 'Diagnosis']], hue='Diagnosis')

for ax in pair.axes.flatten():
    plt.setp(ax.get_xticklabels(), fontsize=14, fontweight='medium')
    plt.setp(ax.get_yticklabels(), fontsize=14, fontweight='regular')
    ax.set_xlabel(ax.get_xlabel(), fontsize=16, fontweight='bold')
    ax.set_ylabel(ax.get_ylabel(), fontsize=16, fontweight='bold')


leg = pair._legend
leg.set_title('Diagnosis', prop={'size': 12, 'weight': 'bold'})
for text in leg.get_texts():
    text.set_fontsize(10)
    text.set_fontweight('bold')

pair.fig.set_dpi(600)
pair.fig.set_size_inches(10, 8)

plt.show()

In [ ]:
#set 1 parameters
sns.pairplot(df[['Age ', 'Diagnosis']], hue='Diagnosis')
plt.figure(figsize=(10, 8),dpi = 600)
plt.show()

sns.pairplot(df[['PLR', 'Diagnosis']], hue='Diagnosis')
plt.figure(figsize=(10, 8),dpi = 600)
plt.show()

sns.pairplot(df[['WBC', 'Diagnosis']], hue='Diagnosis')
plt.figure(figsize=(10, 8),dpi = 600)
plt.show()

sns.pairplot(df[['PLT', 'Diagnosis']], hue='Diagnosis')
plt.figure(figsize=(10, 8),dpi = 600)
plt.show()

sns.pairplot(df[['NE%', 'Diagnosis']], hue='Diagnosis')
plt.figure(figsize=(10, 8),dpi = 600)
plt.show()

In [ ]:
#set  1-2 parameters
sns.pairplot(df[['Age ','PLR', 'Diagnosis']], hue='Diagnosis')
plt.figure(figsize=(10, 8),dpi = 600)
plt.show()

sns.pairplot(df[['WBC','PLT', 'Diagnosis']], hue='Diagnosis')
plt.figure(figsize=(10, 8),dpi = 600)
plt.show()

sns.pairplot(df[['NE%', 'Diagnosis']], hue='Diagnosis')
plt.figure(figsize=(10, 8),dpi = 600)
plt.show()

sns.pairplot(df[['PLT','NE%', 'Diagnosis']], hue='Diagnosis')
plt.figure(figsize=(10, 8),dpi = 600)
plt.show()


In [ ]:
#box plot
plt.figure(figsize=(6, 10), dpi=600)
sns.boxplot(x='Diagnosis', y='Age ', data=df, hue='Diagnosis', palette={'RCC': 'skyblue', 'UTUC': 'orange'})
plt.title('Age Distribution by Diagnosis')
plt.show()

plt.figure(figsize=(6, 10), dpi=600)
sns.boxplot(x='Diagnosis', y='PLR', data=df, hue='Diagnosis', palette={'RCC': 'skyblue', 'UTUC': 'orange'})
plt.title('PLR Distribution by Diagnosis')
plt.show()

plt.figure(figsize=(6, 10), dpi=600)
sns.boxplot(x='Diagnosis', y='WBC', data=df, hue='Diagnosis', palette={'RCC': 'skyblue', 'UTUC': 'orange'})
plt.title('WBC Distribution by Diagnosis')
plt.show()

plt.figure(figsize=(6, 10), dpi=600)
sns.boxplot(x='Diagnosis', y='PLT', data=df, hue='Diagnosis', palette={'RCC': 'skyblue', 'UTUC': 'orange'})
plt.title('PLT Distribution by Diagnosis')
plt.show()

plt.figure(figsize=(6, 10), dpi=600)
sns.boxplot(x='Diagnosis', y='NE%', data=df, hue='Diagnosis', palette={'RCC': 'skyblue', 'UTUC': 'orange'})
plt.title('NE% Distribution by Diagnosis')
plt.show()

# Models comparison and Evaluation Model Performance

In [ ]:
# Models and their names
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=25),
    "Logistic Regression": LogisticRegression(max_iter=200,random_state=25),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Multi-layer Perceptron ": MLPClassifier(max_iter=200,random_state=25),
    "Random Forest": RandomForestClassifier(random_state=25),
    "Gradient Boosting": GradientBoostingClassifier(random_state=25)
}

# Dictionary to store ROC data
roc_data = {}

# Iterate through models
for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Handle hyperparameter tuning for specific models
    if model_name == "Decision Tree":
        param_grid_dt = { # Use a separate param_grid for DecisionTreeClassifier
            'criterion': ['gini', 'entropy'],
            'max_depth': [3, 5, 10],
            'min_samples_split': [2, 10, 20],
            'min_samples_leaf': [2, 5, 10],
            'max_features': [None, 'sqrt', 'log2']
        }
        grid_search_dt = GridSearchCV(model, param_grid_dt, cv=5, scoring='accuracy') # Pass the correct param_grid_dt
        grid_search_dt.fit(X_train_res, y_train_res)
        model = grid_search_dt.best_estimator_
    elif model_name == "Logistic Regression":
        param_grid_ls = { # Corrected parameter grid for Logistic Regression
            'penalty': ['l1', 'l2'],
            'C': [0.001, 0.01, 0.1, 1, 10, 100],
            'solver': ['liblinear', 'saga']
        }
        grid_search_ls = GridSearchCV(model, param_grid_ls, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)
        grid_search_ls.fit(X_train_res, y_train_res)
        model = grid_search_ls.best_estimator_
    elif model_name == "K-Nearest Neighbors":
        param_grid_knn = {
            'n_neighbors': [5, 7, 9, 11, 15],
            'weights': ['uniform'],
            'p': [1, 2, 3]
        }
        grid_search_knn = GridSearchCV(model, param_grid_knn, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)
        grid_search_knn.fit(X_train_res, y_train_res)
        model = grid_search_knn.best_estimator_
    elif model_name == "Multi-layer Perceptron ":
        param_grid_mlp = {
            'hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 50)],
            'activation': ['tanh', 'relu'],
            'solver': ['sgd', 'adam'],
            'alpha': [0.0001, 0.05],
            'learning_rate': ['constant','adaptive']
        }
        grid_search_mlp = GridSearchCV(model, param_grid_mlp, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)
        grid_search_mlp.fit(X_train_res, y_train_res)
        model = grid_search_mlp.best_estimator_
    elif model_name == "Random Forest":
        param_grid_rf = {
            'n_estimators': [50, 100, 200],
            'max_depth': [None, 10, 20, 30],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2']
        }
        grid_search_rf = GridSearchCV(model, param_grid_rf, cv=5, scoring='accuracy', n_jobs=-1, verbose=0) # Pass the correct param_grid_rf
        grid_search_rf.fit(X_train_res, y_train_res)
        model = grid_search_rf.best_estimator_

    elif model_name == "Gradient Boosting":
        param_grid_gb = {
            'n_estimators': [30 , 50, 100],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [2, 3 ,4],
            'min_samples_split': [6, 8, 10],
            'min_samples_leaf': [2, 3, 4]
        }
        grid_search_gb = GridSearchCV(model, param_grid_gb, cv=5, scoring='accuracy', verbose=0) # Pass the correct param_grid_gb
        grid_search_gb.fit(X_train_res, y_train_res)
        model = grid_search_gb.best_estimator_

    # Train the model
    model.fit(X_train_res, y_train_res)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_test_numeric = y_test.map({'RCC': 0, 'UTUC': 1})
    fpr, tpr, thresholds = roc_curve(y_test_numeric, y_prob)
    roc_auc = roc_auc_score(y_test_numeric, y_prob)
    roc_data[model_name] = (fpr, tpr, roc_auc)

# Plotting the ROC curves
plt.figure(figsize=(10, 8),dpi = 600)
for model_name, (fpr, tpr, roc_auc) in roc_data.items():
    plt.plot(fpr, tpr, lw=2, label=f'{model_name} (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curves')
plt.legend(loc="lower right")
plt.show()

## Differential_Diagnosis_RCC_UTUC_ each model

Decision Tree

In [ ]:
# Train Final Model ด้วย Optimal Features
# ============================================================
best_model_dt = RFE(
    estimator=DecisionTreeClassifier(random_state=25)

)

# Fit RFE on training set
best_model_dt.fit(X_train_res, y_train_res)


In [ ]:
# Evaluate the Decision Tree model
best_model_dt.fit(X_train_res, y_train_res)

y_pred_dt = best_model_dt.predict(X_test)
# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred_dt)
print(f"Accuracy: {accuracy}")

# Calculate Specificity
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_dt).ravel()
specificity_dt = tn / (tn + fp)
print(f"Decision Tree Specificity: {specificity_dt}")

print("Classification Report:")
print(classification_report(y_test, y_pred_dt))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt))

In [ ]:
#Confusion Matrix for Decision Tree
cm = confusion_matrix(y_test, y_pred_dt)

plt.figure(figsize=(8, 6),dpi=600)
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    annot_kws={"size":28, "weight":"bold"},
    xticklabels=['RCC', 'UTUC'], yticklabels=['RCC', 'UTUC'])

plt.xlabel('Predicted', fontsize=16, fontweight='bold')
plt.ylabel('Actual', fontsize=16, fontweight='bold')
plt.title('Decision Tree Model', fontsize=20, fontweight='bold')

plt.xticks(fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')
plt.show()


Logistic Regression

In [ ]:
# Train Final Model by Optimal Features
# ============================================================
best_model_lr = RFE(
    estimator=LogisticRegression(random_state=25)

)

# Fit RFE on training set ท
best_model_lr.fit(X_train_res, y_train_res)


In [ ]:
# Evaluate the Logistic Regression model
best_model_lr.fit(X_train_res, y_train_res)

y_pred_lr = best_model_lr.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred_lr)
print(f"Accuracy: {accuracy}")

# Calculate Specificity
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_lr).ravel()
specificity_lr = tn / (tn + fp)
print(f"Logistic Regression Specificity: {specificity_lr}")

print("Classification Report:")
print(classification_report(y_test, y_pred_lr))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

In [ ]:
#Confusion Matrix for Logistic Regression
cm = confusion_matrix(y_test, y_pred_lr)

plt.figure(figsize=(8, 6),dpi=600)
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    annot_kws={"size":28, "weight":"bold"},
    xticklabels=['RCC', 'UTUC'], yticklabels=['RCC', 'UTUC'])

plt.xlabel('Predicted', fontsize=16, fontweight='bold')
plt.ylabel('Actual', fontsize=16, fontweight='bold')
plt.title('Logistic Regression Model', fontsize=20, fontweight='bold')

plt.xticks(fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')
plt.show()

K-Nearest Neighbors

In [ ]:
# Train Final Model ด้วย Optimal Features
# ============================================================
best_model_knn = KNeighborsClassifier(n_neighbors=10)

# Apply the feature selection from final_rfe before fitting
X_train_res_rfe = final_rfe.transform(X_train_res)

# Fit the KNeighborsClassifier on the transformed training set
best_model_knn.fit(X_train_res_rfe, y_train_res)

In [ ]:
# Evaluate K-Nearest Neighbors  model
best_model_knn.fit(X_train_res, y_train_res)
# Predictions
y_pred_knn = best_model_knn.predict(X_test)

# Calculate accuracy
accuracy_knn = accuracy_score(y_test, y_pred_knn)
print(f"Accuracy: {accuracy_knn}")

# Calculate Specificity
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_knn).ravel()
specificity_knn = tn / (tn + fp)
print(f"K-Nearest Neighbors Specificity: {specificity_knn}")

print("Classification Report:")
print(classification_report(y_test, y_pred_knn))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_knn))

In [ ]:
#Confusion Matrix for K-Nearest Neighbors
cm = confusion_matrix(y_test, y_pred_knn)

plt.figure(figsize=(8, 6),dpi=600)
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    annot_kws={"size":28, "weight":"bold"},
    xticklabels=['RCC', 'UTUC'], yticklabels=['RCC', 'UTUC'])

plt.xlabel('Predicted', fontsize=16, fontweight='bold')
plt.ylabel('Actual', fontsize=16, fontweight='bold')
plt.title('K-Nearest Neighbors Model', fontsize=20, fontweight='bold')

plt.xticks(fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')
plt.show()

MLPClassifier

In [ ]:
# Train Final Model ด้วย Optimal Features
# ============================================================
best_model_mlp = MLPClassifier(max_iter=200,random_state=25)

# Apply the feature selection from final_rfe before fitting
X_train_res_rfe = final_rfe.transform(X_train_res)

# Fit the MLPClassifier on the transformed training set
best_model_mlp.fit(X_train_res_rfe, y_train_res)

In [ ]:
# Evaluate  Multi-layer Perceptron  model
best_model_mlp.fit(X_train_res, y_train_res)

y_pred_mlp = best_model_mlp.predict(X_test)

# Calculate accuracy
accuracy_mlp = accuracy_score(y_test, y_pred_mlp)
print(f"Accuracy: {accuracy_mlp}")

# Calculate Specificity
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_mlp).ravel()
specificity_mlp = tn / (tn + fp)
print(f"Multi-layer Perceptron  Specificity: {specificity_mlp}")

print("Classification Report:")
print(classification_report(y_test, y_pred_mlp))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_mlp))

In [ ]:
#Confusion Matrix for Multi-layer Perceptron
cm = confusion_matrix(y_test, y_pred_mlp)

plt.figure(figsize=(8, 6),dpi=600)
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    annot_kws={"size":28, "weight":"bold"},
    xticklabels=['RCC', 'UTUC'], yticklabels=['RCC', 'UTUC'])

plt.xlabel('Predicted', fontsize=16, fontweight='bold')
plt.ylabel('Actual', fontsize=16, fontweight='bold')
plt.title('Multi-layer Perceptron Model', fontsize=20, fontweight='bold')

plt.xticks(fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')
plt.show()

Random Forest

In [ ]:
# Evaluate Random Forest model
final_rf.fit(X_train_res, y_train_res)
# Make predictions on the test set
y_pred_rf = final_rf.predict(X_test)

# Evaluate the model
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f"Accuracy: {accuracy_rf}")

# Calculate Specificity
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_rf).ravel()
specificity_rf = tn / (tn + fp)
print(f"Random Forest Specificity: {specificity_rf}")

print("Classification Report:")
print(classification_report(y_test, y_pred_rf))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))



In [ ]:
#Confusion Matrix for Random Forest model
cm = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(8, 6),dpi=600)
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    annot_kws={"size":28, "weight":"bold"},
    xticklabels=['RCC', 'UTUC'], yticklabels=['RCC', 'UTUC'])

plt.xlabel('Predicted', fontsize=16, fontweight='bold')
plt.ylabel('Actual', fontsize=16, fontweight='bold')
plt.title('Random Forest Model', fontsize=20, fontweight='bold')

plt.xticks(fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')
plt.show()

 Gradient Boosting classifier

In [ ]:
# Train Final Model by Optimal Features
# ============================================================
best_model_gb = RFE(
    estimator=GradientBoostingClassifier(random_state=25)

)

# Fit RFE on training set
best_model_gb.fit(X_train_res, y_train_res)


In [ ]:
# Evaluate the Gradient Boosting model
best_model_gb.fit(X_train_res, y_train_res)
y_pred_gb = best_model_gb.predict(X_test)

# Calculate accuracy
accuracy_gb = accuracy_score(y_test, y_pred_gb)
print(f"Accuracy: {accuracy_gb}")

# Calculate Specificity
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_gb).ravel()
specificity_gb = tn / (tn + fp)
print(f"Gradient Boosting Specificity: {specificity_gb}")

print("Classification Report:")
print(classification_report(y_test, y_pred_gb))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_gb))

In [ ]:
#Confusion Matrix for Gradient Boosting model
cm = confusion_matrix(y_test, y_pred_gb)


plt.figure(figsize=(8, 6),dpi=600)
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    annot_kws={"size":28, "weight":"bold"},
    xticklabels=['RCC', 'UTUC'], yticklabels=['RCC', 'UTUC'])

plt.xlabel('Predicted', fontsize=16, fontweight='bold')
plt.ylabel('Actual', fontsize=16, fontweight='bold')
plt.title('Gradient Boosting model', fontsize=20, fontweight='bold')

plt.xticks(fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')
plt.show()


In [ ]:
print("\n--- Decision Tree Classification Report ---")
print(classification_report(y_test, y_pred_dt, target_names=['RCC', 'UTUC']))

print("\n--- Logistic Regression Classification Report ---")
print(classification_report(y_test, y_pred_lr, target_names=['RCC', 'UTUC']))

print("\n--- K-Nearest Neighbors Classification Report ---")
print(classification_report(y_test, y_pred_knn, target_names=['RCC', 'UTUC']))

print("\n--- Multi-layer Perceptron Classification Report ---")
print(classification_report(y_test, y_pred_mlp, target_names=['RCC', 'UTUC']))

print("\n--- Random Forest Classification Report ---")
print(classification_report(y_test, y_pred_rf, target_names=['RCC', 'UTUC']))

print("\n--- Gradient Boosting Classification Report ---")
print(classification_report(y_test, y_pred_gb, target_names=['RCC', 'UTUC']))